# 🎯 Notebook 2: SLI, SLO, SLA — and the Error Budget

Three letters that confuse everyone:

- **SLI** — Service Level **Indicator**. The number you measure. *e.g. "% of requests that returned 200 in under 300 ms"*.
- **SLO** — Service Level **Objective**. The target you set internally. *e.g. "99.9% over a rolling 28 days"*.
- **SLA** — Service Level **Agreement**. The contractual promise to a customer (with money or service credits if you miss it). Usually looser than the SLO so you have headroom.

Out of this falls the **error budget**: if your SLO is 99.9%, you are *allowed* to be bad **0.1%** of the time. That 0.1% is a budget you can spend on risky deploys and experiments.

## Learning objectives
- Tell **vanity metrics** from **user-visible SLIs**.
- Compute an SLI from raw request data and check it against an SLO.
- Track an error budget over time, and learn what **fast burn vs slow burn** look like.
- Pick metrics with the **RED method** (and know about USE for resources).

## 🪞 Vanity metrics vs user-visible SLIs

Beginner trap: alerting on metrics that *look* important but don't reflect what users feel.

| ❌ Vanity (machine-centric) | ✅ User-visible SLI |
|---|---|
| `cpu_percent` is high | `% of checkout requests succeeding < 300 ms` |
| `disk_io` is busy | `% of search results returned in < 1 s` |
| `gc_pause_ms` | `% of video starts within 2 s` |

CPU at 90% might be fine if every user is happy. CPU at 30% is **not** fine if half your users are getting 504s.

**Rule of thumb:** an SLI should be something you can put in a sentence a non-engineer understands.

## 🚦 The RED method (and a word on USE)

When you're staring at a blank dashboard for a new service, what do you measure first?

**RED** — for *services that handle requests* (APIs, web apps):
- **R**ate — requests per second
- **E**rrors — failed requests per second (or %)
- **D**uration — how long requests take (p50/p95/p99)

**USE** — for *resources* (CPU, disk, network, DB pool):
- **U**tilization — % time the resource is busy
- **S**aturation — work queued up waiting for the resource
- **E**rrors — error events from the resource

These are the **Four Golden Signals** in slightly different clothes (Google SRE adds *Saturation* to RED). Start every dashboard with these — *then* add domain-specific things.

## 🧪 Computing an SLI from raw data

Let's simulate one minute of traffic — 1000 requests — and compute the SLI: *"% of requests that succeeded in under 300 ms."*

In [ ]:
import random
random.seed(42)

requests = []
for _ in range(1000):
    latency = max(0, random.gauss(150, 80))
    success = (latency < 500) and (random.random() > 0.002)  # ~99.8% success
    requests.append({"latency_ms": latency, "ok": success})

LATENCY_THRESHOLD_MS = 300
good = sum(1 for r in requests if r["ok"] and r["latency_ms"] < LATENCY_THRESHOLD_MS)
total = len(requests)
sli = good / total

print(f"SLI (% requests OK and < {LATENCY_THRESHOLD_MS} ms): {sli:.1%}")
print(f"SLO target: 99.0%  ->  {'✅ on track' if sli >= 0.99 else '❌ missed'}")

### 🤔 We missed — so was the service broken?

No. Almost every request *succeeded*; the SLI failed on the **latency** half of the
definition. `random.gauss(150, 80)` puts roughly 3% of requests over 300 ms, so a
99% SLO against a 300 ms threshold was never achievable for this service. Nothing
was broken; the objective was wishful thinking.

This is the most common way SLOs go wrong: someone picks a round number ("99.9%!
under 300 ms!") before measuring anything, the SLO burns its budget on week one,
and the team learns to ignore it.

**Set the threshold from the distribution you actually have**, then work to improve
it. Let's find the threshold this service can genuinely hold at 99%.

In [ ]:
import math

ok_latencies = sorted(r["latency_ms"] for r in requests if r["ok"])
error_count = sum(1 for r in requests if not r["ok"])

# To hit a 99% SLO we need ceil(99% x total) requests that are BOTH ok and fast.
# Errors already eat into that count, so the threshold is set by the (990th) fastest
# successful request, not by the p99 of the successes.
needed = math.ceil(0.99 * total)
achievable = ok_latencies[needed - 1]
print(f"errors: {error_count}/{total}   "
      f"need {needed} good+fast requests   -> threshold {achievable:.0f} ms")

for threshold in (300, achievable, 400):
    hit = sum(1 for r in requests if r["ok"] and r["latency_ms"] <= threshold) / total
    verdict = "✅" if hit >= 0.99 else "❌"
    print(f"  threshold {threshold:6.0f} ms -> SLI {hit:6.1%}  {verdict} against a 99% SLO")

# By construction this threshold is the tightest one the service can actually hold.
final = sum(1 for r in requests if r["ok"] and r["latency_ms"] <= achievable) / total
assert final >= 0.99, final
tighter = sum(1 for r in requests if r["ok"] and r["latency_ms"] < achievable) / total
assert tighter < 0.99, "should be the *tightest* threshold that still passes"
assert achievable > 300, "300 ms was never reachable for this service"
print(f"\nAn honest starting SLO for this service: 99% of requests OK "
      f"in under ~{achievable:.0f} ms — then work the number down.")

## 💰 The error budget

If your SLO is 99.0% over 30 days and you serve 1,000,000 requests in that window, your budget is:

```
budget = total × (1 − SLO) = 1_000_000 × 0.01 = 10_000 bad requests
```

Many teams use this rule:
> **If the budget is being burnt too fast, freeze risky deploys and focus on reliability. If the budget is unused, take more risks (chaos tests, faster releases).**

In [ ]:
TOTAL_REQUESTS = 1_000_000
SLO = 0.99
budget = TOTAL_REQUESTS * (1 - SLO)
print(f"30-day error budget: {int(budget):,} bad requests\n")

# Simulate 30 days, with one really bad day in the middle.
spent = 0
for day in range(1, 31):
    bad_today = 200 if day != 15 else 6000  # a big outage on day 15
    spent += bad_today
    pct = spent / budget
    bar = "█" * min(40, int(pct * 40))
    flag = "🚨" if pct > 1 else ("⚠️ " if pct > 0.7 else "  ")
    print(f"day {day:2d}: spent {spent:6d}/{int(budget)}  {bar:40s} {flag}")

print(f"\nSpent {spent:,} of a {int(budget):,} budget => {spent/budget:.0%}.")
assert spent > budget, "the day-15 outage should have blown the 30-day budget"
print("One 6,000-error day cost more than the other 29 days combined —")
print("which is exactly why you alert on burn *rate*, not on a monthly total.")

## 🔥 Burn rate — fast burn vs slow burn

The **burn rate** is *how fast you're spending the budget* relative to "evenly over 30 days".

- Burn rate = **1.0** → you'll exactly use up the budget by day 30 (on plan).
- Burn rate = **10** → at this pace you'll exhaust the budget in 3 days. Page someone now.
- Burn rate = **0.5** → you have headroom; ship that risky feature.

Real SRE teams alert on **two** windows at once so they catch both kinds of trouble.
The thresholds below are the ones from the Google SRE workbook, and they are not
arbitrary — each is "the rate at which you would burn X% of a 30-day budget in this
window":

| Kind | Windows | Threshold | Budget burnt before it fires | Response |
|---|---|---|---|---|
| **Fast burn** | 1 h **and** 5 m | `> 14.4×` | 2% | page on-call |
| **Medium burn** | 6 h **and** 30 m | `> 6×` | 5% | page on-call |
| **Slow burn** | 3 d **and** 6 h | `> 1×` | 10% | open a ticket |

The short second window is there to stop the alert *resolving* slowly once the
incident is over. Below we compute the burn rate for three scenarios and check each
against the right threshold.

In [ ]:
# Helper: burn rate = (errors in window / requests in window) / (1 - SLO)
def burn_rate(errors, requests, slo):
    if requests == 0:
        return 0.0
    error_rate = errors / requests
    allowed_error_rate = 1 - slo
    return error_rate / allowed_error_rate

SLO = 0.99
RPS = 100  # requests per second

FAST_THRESHOLD, MEDIUM_THRESHOLD, SLOW_THRESHOLD = 14.4, 6.0, 1.0

# Scenario A: a hard outage right now — 20% of requests failing for 5 minutes.
fast_reqs = RPS * 5 * 60
fast_errs = int(fast_reqs * 0.20)
fast = burn_rate(fast_errs, fast_reqs, SLO)
print(f"OUTAGE:    {fast_errs:>7} errors in {fast_reqs:>7} reqs over 5 min")
print(f"  burn rate = {fast:5.1f}x  (fast-burn threshold {FAST_THRESHOLD}x)  -> 🚨 page on-call")
assert fast > FAST_THRESHOLD

# Scenario B: a small regression that has been running for 6 hours — 1.2% errors.
slow_reqs = RPS * 6 * 3600
slow_errs = int(slow_reqs * 0.012)
slow = burn_rate(slow_errs, slow_reqs, SLO)
print(f"\nREGRESSION:{slow_errs:>7} errors in {slow_reqs:>7} reqs over 6 h")
print(f"  burn rate = {slow:5.1f}x  -> below {MEDIUM_THRESHOLD}x so nobody is paged,")
print(f"                  but above {SLOW_THRESHOLD}x -> ⚠️  ticket: this will eat the budget")
assert SLOW_THRESHOLD < slow < MEDIUM_THRESHOLD

# Scenario C: healthy — 0.5% errors, comfortably inside the budget.
healthy_errs = int(slow_reqs * 0.005)
healthy = burn_rate(healthy_errs, slow_reqs, SLO)
print(f"\nHEALTHY:   {healthy_errs:>7} errors in {slow_reqs:>7} reqs over 6 h")
print(f"  burn rate = {healthy:5.1f}x  -> ✅ under 1x, the budget is regenerating faster "
      f"than we spend it")
assert healthy < SLOW_THRESHOLD

# Sanity: a 14.4x burn really does eat 2% of a 30-day budget in one hour.
budget_fraction_per_hour = FAST_THRESHOLD * (1 / (30 * 24))
print(f"\n(Check: {FAST_THRESHOLD}x for 1 h burns "
      f"{budget_fraction_per_hour:.1%} of a 30-day budget.)")
assert abs(budget_fraction_per_hour - 0.02) < 0.001

**Why two windows?** A single short window catches loud outages but flaps on noise. A single long window catches slow regressions but reacts too late to a fire. Combining them gives you alerts that are both *fast* and *quiet*.

## 🤝 SLA vs SLO — why the gap?

People often ask, *"why are these different?"*

- **SLO** is what you aim for internally (e.g. 99.95%). You want headroom.
- **SLA** is what you promise customers in a contract (e.g. 99.9%). Missing it costs you money — service credits, refunds, reputational damage.

**Always set your SLO tighter than your SLA**, so you start sweating *before* you owe customers money.

## ✅ Recap

- **Pick SLIs that reflect what users feel**, not what your machines feel.
- Use the **RED** method for services and **USE** for resources to start any dashboard.
- The error budget turns reliability from a vibe into **math** — and the **burn rate** tells you *how urgent* a problem is.
- Alert on **two windows** (fast + slow) for fewer false alarms.

In the next notebook we build the actual collector that produces the numbers behind all of this.